In [0]:
from pyspark.sql.functions import col, upper, trim, regexp_replace, translate, split

# 1. Read raw data from the Bronze Managed Table
table_name_bronze = "bronze_ibge_population"
df_bronze = spark.read.table(table_name_bronze)

# 2. Filter the IBGE Header Row 
# Removing the first row that contains the API column descriptions
df_filtered = df_bronze.filter(col("D1C") != "Município (Código)")

# 3. Data Cleansing and Standardization
accents = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇáàâãäéèêëíìîïóòôõöúùûüç"
no_accents = "AAAAAEEEEIIIIOOOOOUUUUCAAAAAEEEEIIIIOOOOOUUUUC"

# 3.1 Extract only the city name (removing the " - UF" suffix)
df_silver = df_filtered.withColumn(
    "municipality_name_clean", 
    split(col("D1N"), " - ").getItem(0)
)

# 3.2 Remove accents, convert to uppercase, and trim spaces
df_silver = df_silver.withColumn(
    "municipality_name_clean", 
    upper(trim(translate(col("municipality_name_clean"), accents, no_accents)))
)

# 3.3 Remove any remaining special characters (keep only letters, numbers, and spaces)
df_silver = df_silver.withColumn(
    "municipality_name_clean",
    regexp_replace(col("municipality_name_clean"), "[^A-Z0-9 ]", "")
)

# 4. Rename columns to standard English snake_case and cast types
df_silver = df_silver.withColumnRenamed("D1C", "municipality_code") \
                     .withColumnRenamed("D1N", "municipality_name_original") \
                     .withColumnRenamed("V", "population_estimated")

# Cast population to integer type for future calculations
df_silver = df_silver.withColumn("population_estimated", col("population_estimated").cast("integer"))

# Select only the relevant columns for the analytical model
df_silver_final = df_silver.select(
    "municipality_code", 
    "municipality_name_original",
    "municipality_name_clean", 
    "population_estimated", 
    "ingestion_timestamp",
    "source_system"
)

# 5. Save to the Silver Layer as a Managed Delta Table
table_name_silver = "silver_ibge_population_cleansed"

df_silver_final.write \
               .format("delta") \
               .mode("overwrite") \
               .saveAsTable(table_name_silver)

print(f"Silver processing complete! {df_silver_final.count()} records cleaned and saved to table '{table_name_silver}'.")
display(df_silver_final)